# 🏃‍♂️ Exploratory Data Analysis (EDA) — Human Activity Recognition

This notebook visualizes and analyzes tri-axial accelerometer signals from wearable sensors. It explores temporal waveforms, 3D trajectory plots, signal statistics, and dimensionality reduction via Principal Component Analysis (PCA) to distinguish stationary activities (Laying, Sitting, Standing) from dynamic ones (Walking, Walking Upstairs, Walking Downstairs).

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Ensure project root is in the path for src imports
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils.helpers import load_config
config = load_config("config/config.yaml")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

## 📊 Data Loading & Verification

We load accelerometer data. If the raw UCI HAR Dataset is not found locally, the notebook automatically falls back to generating realistic mock sensor signals so that all visualizations and downstream analyses remain fully executable.

In [ ]:
combined_dir = os.path.join(PROJECT_ROOT, config["data"]["combined_dir"])
activities = list(config["activities"].values())
has_real_data = os.path.exists(combined_dir)

if has_real_data:
    print("✅ Real combined dataset found. Loading signals...")
    # Load a sample signal per activity
    signals = {}
    for act in activities:
        act_path = os.path.join(combined_dir, "Train", act)
        if os.path.exists(act_path):
            files = sorted(os.listdir(act_path))
            if files:
                df = pd.read_csv(os.path.join(act_path, files[0]))
                signals[act] = df.values[:500]  # Take first 500 samples
else:
    print("⚠️ Combined dataset not found. Generating mock accelerometer signals for demonstration...")
    # Generate mock signals representing typical activity patterns
    np.random.seed(42)
    t = np.linspace(0, 10, 500)
    signals = {}
    
    # Laying: stationary, aligned with one axis (e.g., Z has gravity)
    signals["LAYING"] = np.stack([
        np.random.normal(0, 0.05, 500), # X
        np.random.normal(0, 0.05, 500), # Y
        np.ones(500) + np.random.normal(0, 0.05, 500) # Z (gravity)
    ], axis=1)
    
    # Sitting: stationary, different orientation
    signals["SITTING"] = np.stack([
        np.random.normal(0, 0.05, 500),
        np.ones(500) * 0.9 + np.random.normal(0, 0.05, 500),
        np.ones(500) * 0.4 + np.random.normal(0, 0.05, 500)
    ], axis=1)
    
    # Standing: stationary, upright
    signals["STANDING"] = np.stack([
        np.random.normal(0, 0.08, 500),
        np.ones(500) * 0.95 + np.random.normal(0, 0.08, 500),
        np.random.normal(0, 0.08, 500)
    ], axis=1)
    
    # Walking: regular periodic pattern
    signals["WALKING"] = np.stack([
        0.4 * np.sin(2 * np.pi * 1.5 * t) + np.random.normal(0, 0.15, 500),
        0.95 * np.ones(500) + 0.3 * np.cos(2 * np.pi * 1.5 * t) + np.random.normal(0, 0.15, 500),
        0.2 * np.sin(2 * np.pi * 1.5 * t) + np.random.normal(0, 0.15, 500)
    ], axis=1)
    
    # Walking Upstairs: periodic, offset due to climbing effort
    signals["WALKING_UPSTAIRS"] = np.stack([
        0.5 * np.sin(2 * np.pi * 1.8 * t) + np.random.normal(0, 0.2, 500),
        0.8 * np.ones(500) + 0.4 * np.cos(2 * np.pi * 1.8 * t) + np.random.normal(0, 0.2, 500),
        0.3 * np.sin(2 * np.pi * 1.8 * t) + 0.1 * t + np.random.normal(0, 0.2, 500)
    ], axis=1)
    
    # Walking Downstairs: faster, more high-frequency impacts
    signals["WALKING_DOWNSTAIRS"] = np.stack([
        0.6 * np.sin(2 * np.pi * 2.0 * t) + np.random.normal(0, 0.25, 500),
        1.1 * np.ones(500) + 0.5 * np.cos(2 * np.pi * 2.0 * t) + np.random.normal(0, 0.25, 500),
        0.4 * np.sin(2 * np.pi * 2.0 * t) + np.random.normal(0, 0.25, 500)
    ], axis=1)

## 📈 Waveform Analysis

We plot the X, Y, and Z accelerometer components to observe signal frequencies and patterns.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 12), sharex=True)
axes = axes.flatten()

for idx, act in enumerate(activities):
    sig = signals[act]
    ax = axes[idx]
    ax.plot(sig[:, 0], label="X-axis", alpha=0.8)
    ax.plot(sig[:, 1], label="Y-axis", alpha=0.8)
    ax.plot(sig[:, 2], label="Z-axis", alpha=0.8)
    ax.set_title(act)
    ax.set_ylabel("Acceleration (g)")
    if idx >= 4:
        ax.set_xlabel("Time Steps")
    if idx == 0:
        ax.legend()

plt.suptitle("Accelerometer Waveforms by Activity", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

## 🚀 Total Acceleration Magnitude ($||acc||$)

Static activities should show a constant magnitude close to $1.0\text{g}$ (the gravity vector), whereas dynamic activities should show high variance.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=True)
axes = axes.flatten()

for idx, act in enumerate(activities):
    sig = signals[act]
    mag = np.sqrt(np.sum(sig**2, axis=1))
    
    ax = axes[idx]
    ax.plot(mag, color="purple")
    ax.set_title(act)
    ax.set_xlabel("Time Steps")
    if idx % 3 == 0:
        ax.set_ylabel("Magnitude (g)")

plt.suptitle("Total Acceleration Magnitude ($||acc||$)", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

## 📐 3D Trajectory Plots

Visualizing the trajectory in 3D acceleration space reveals structural differences in movement patterns.

In [ ]:
fig = plt.subplots(figsize=(18, 10))

for idx, act in enumerate(activities):
    sig = signals[act]
    ax = plt.subplot(2, 3, idx + 1, projection='3d')
    ax.plot(sig[:, 0], sig[:, 1], sig[:, 2], linewidth=1.0, color="teal")
    ax.set_title(act)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.grid(True)

plt.suptitle("3D Trajectories in Acceleration Space", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

## 🧪 Dimensionality Reduction via PCA

We extract simple mean vectors from multiple windows to check if simple static characteristics can separate the classes under a PCA projection.

In [ ]:
# Build a dataset of mean acceleration vectors
X_list = []
y_list = []
np.random.seed(42)

for act in activities:
    base_sig = signals[act]
    # Generate 50 synthetic window variations by slicing or adding noise to the base signal
    for _ in range(50):
        variant = base_sig + np.random.normal(0, 0.05, base_sig.shape)
        X_list.append(np.mean(variant, axis=0))
        y_list.append(act)

X_mean = np.array(X_list)
y_labels = np.array(y_list)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_mean)

plt.figure(figsize=(10, 8))
for act in activities:
    idx = y_labels == act
    plt.scatter(X_pca[idx, 0], X_pca[idx, 1], label=act, s=50, alpha=0.8)

plt.title("2D PCA Projection of Mean Acceleration Vectors")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()
plt.grid(True)
plt.show()